# Data preparation

Statistics and checks for `src/data/prepare.py`. Run `python src/data/prepare.py` first.

In [1]:
import sys
from pathlib import Path

sys.path.append("..")

import pandas as pd
from src.data.prepare import OUT_DIR, RAW_CSV, SOURCES

## Raw data

Vision4Chart mixes chart and non-chart sources. We keep ChartQA, FigureQA and DVQA.

In [2]:
raw = pd.read_csv(RAW_CSV, usecols=["caption", "neg_caption", "img_path"])
raw["image"] = raw.img_path.str.removeprefix("./")
raw["source"] = raw.image.str.split("/").str[2]
raw.source.value_counts()

source
dvqa             2000000
plotqa           2000000
figureqa         1000000
ocrvqa            801579
clevr             699989
ChartBench        568475
vqav2             443757
robut_wikisql     288957
chartqa           169034
mapqa             151536
chart2text         87946
docvqa             39463
textvqa            34602
aokvqa             17056
Name: count, dtype: int64

Rows per source and rows removed: empty statements, identical correct/incorrect statements, and duplicates.

In [3]:
charts = raw[raw.source.isin(SOURCES)].copy()
charts["caption"] = charts.caption.str.strip()
charts["neg_caption"] = charts.neg_caption.str.strip()

empty = charts.caption.isna() | charts.neg_caption.isna()
identical = ~empty & (charts.caption == charts.neg_caption)
duplicate = charts.duplicated(["image", "caption", "neg_caption"])

pd.DataFrame({
    "rows": charts.groupby("source").size(),
    "charts": charts.groupby("source").image.nunique(),
    "empty": empty.groupby(charts.source).sum(),
    "identical": identical.groupby(charts.source).sum(),
    "duplicate": duplicate.groupby(charts.source).sum(),
})

,rows,charts,empty,identical,duplicate
source,,,,,
chartqa,169034,18317,4,34,189
dvqa,2000000,200000,12,19928,0
figureqa,1000000,99992,0,4,0


## Splits

`CHARTS_PER_SOURCE` charts are sampled from each source and split 70/15/15 by chart, so all statements of a chart stay in the same split.

In [4]:
splits = {name: pd.read_csv(OUT_DIR / f"{name}.csv") for name in ["train", "val", "test"]}

pd.DataFrame({
    name: {"instances": len(df), "charts": df.image.nunique(), "label_1_share": df.label.mean()}
    for name, df in splits.items()
}).T

,instances,charts,label_1_share
train,61321.0,6300.0,0.495850
val,13315.0,1350.0,0.502666
test,13123.0,1350.0,0.503010


In [5]:
images = {name: set(df.image) for name, df in splits.items()}
assert not images["train"] & images["val"]
assert not images["train"] & images["test"]
assert not images["val"] & images["test"]

## Categories

Assigned by keyword rules (`category` in `prepare.py`). Categories are strongly tied to the source, so RQ3 results should also be reported per source.

In [6]:
test = splits["test"]
test["statement"] = test.option_0.where(test.label == 0, test.option_1)
pd.crosstab(test.source, test.category, margins=True)

category,comparison,lookup,other,All
source,,,,
chartqa,488,3692,32,4212
dvqa,2111,715,1514,4340
figureqa,4350,0,221,4571
All,6949,4407,1767,13123


In [7]:
for name, group in test.groupby("category"):
    print(f"## {name}")
    print("\n".join(group.statement.sample(5, random_state=0)), "\n")

## comparison
Brown is not greater than Red.
90% of people like the most preferred object in the whole chart.
Blue is not the low median.
Cornflower is not the maximum.
Orchid has the lowest value. 

## lookup
The American customer satisfaction index scores of McDonald's restaurants in the United States in 2020 were 70.
Thirty-three point four percent of public schools recorded a theft.
The difference in the Cassava yield of Brazil and the average yield of all the countries/regions is 3.792.
The number of mobile internet users in Saudi Arabia in 2019 was 28.8.
The percentage of expenditure on fresh fruits is 36.73. 

## other
The bars are horizontal.
The blue color represents a tent.
The label of the second bar from the left in each group is "judge".
The forestgreen color represents a bird.
The label of the second group of bars from the left is fiber. 



## Manual review of category rules

30 test statements per category are written to `category_review.csv`. Fill the `human` column by hand (`lookup`, `comparison` or `other`) and re-run to measure how often the rules agree.

In [8]:
REVIEW = Path("category_review.csv")

if not REVIEW.exists():
    sample = test.groupby("category").sample(30, random_state=0)[["source", "statement", "category"]]
    sample["human"] = ""
    sample.to_csv(REVIEW, index=False)

review = pd.read_csv(REVIEW).dropna(subset=["human"])
print(f"{len(review)} reviewed, agreement: {(review.human == review.category).mean():.1%}")
pd.crosstab(review.category, review.human)

0 reviewed, agreement: nan%


human
category
